In [8]:
import pandas as pd
import rdkit

In [9]:
df_protacs = pd.read_csv('PROTAC-DB-3.0/protac.csv', low_memory=False)
df_linker = pd.read_csv('PROTAC-DB-3.0/linker.csv')
df_protacs = df_protacs[['Compound ID', 'Target', 'E3 ligase', 'Smiles', 'Molecular Weight', 'XLogP3', 'Heavy Atom Count',
                         'Hydrogen Bond Acceptor Count', 'Hydrogen Bond Donor Count', 'Ring Count',
                         'Rotatable Bond Count', 'Topological Polar Surface Area']]
df_linker = df_linker[['Compound ID', 'Smiles', 'Smiles_R']]
df_protacs = df_protacs.rename(columns={'Smiles': 'Smiles_protacs'})
df_linker = df_linker.rename(columns={'Smiles': 'Smiles_linker'})
df_protacs = df_protacs.drop_duplicates(subset=['Compound ID']) # 6107
#  missing ID in PROTAC-DB
cnt = 0
for i in range(1, 6111):
    if i not in df_protacs['Compound ID'].tolist():
        cnt += 1
        print(i)

4654
5714
5715
5716


In [10]:
import pickle
with open('webpages_crawling/linker_to_protac.pkl', 'rb') as handle:
    linker_to_protac = pickle.load(handle)
linker_to_protac = {k: list(set(v)) for k, v in linker_to_protac.items()}

In [11]:
dataset_protac = []
for k, v in linker_to_protac.items():
    row_linker = df_linker[df_linker['Compound ID'] == k]
    row_linker = row_linker[['Smiles_linker', 'Smiles_R']].reset_index(drop=True)
    for ele in v:
        row_protac = df_protacs[df_protacs['Compound ID'] == int(ele)].reset_index(drop=True)
        merged_row = pd.concat([row_protac, row_linker], axis=1)
        dataset_protac.append(merged_row)
dataset_protac = pd.concat(dataset_protac, ignore_index=True)
dataset_protac = dataset_protac.drop_duplicates(subset=['Compound ID'])
dataset_protac.to_csv('dataset_protac.csv', index=False)
len(dataset_protac)

5925

In [12]:
# flaws of linkers' webpages in PROTAC-DB
protac_missing_linker = []
for protac_id in df_protacs['Compound ID']:
    if protac_id not in dataset_protac['Compound ID'].tolist():
        protac_missing_linker.append(protac_id)
len(protac_missing_linker)

182

In [13]:
from rdkit import Chem
from rdkit.Chem import AllChem
def write_sdf(mol, path):
    w =  Chem.SDWriter(path)
    # w.SetKekulize(False)
    w.write(mol)
    w.close()

def smi2sdf(smi, sdf_path):
    m1 = Chem.MolFromSmiles(smi)
    Chem.Kekulize(m1)
    m2 = Chem.AddHs(m1)
    tmp = AllChem.EmbedMolecule(m2, useRandomCoords=True)
    if tmp < 0:
        print(f'{sdf_path} failed')
    else:
        AllChem.MMFFOptimizeMolecule(m2)
        m3 = Chem.RemoveHs(m2)
        write_sdf(m3, sdf_path)

In [14]:
import os
from tqdm import tqdm
path_whole_protac_sdf = 'PROTAC-DB-3.0/protac.sdf'
supplier = Chem.SDMolSupplier(path_whole_protac_sdf)
id2protacs = {mol.GetProp("_Name"): mol for mol in supplier if mol is not None}
print(len(id2protacs))
sdf_path_prefix = 'SDF'

for protac_id, smiles_protac, smiles_linker in \
        tqdm(zip(dataset_protac['Compound ID'], dataset_protac['Smiles_protacs'], dataset_protac['Smiles_linker']), total=len(dataset_protac)):
    path_protac_sdf = os.path.join(sdf_path_prefix, str(protac_id) + '_protac.sdf')
    path_linker_sdf = os.path.join(sdf_path_prefix, str(protac_id) + '_linker.sdf')
    os.makedirs(sdf_path_prefix, exist_ok=True)
    mol_protac = id2protacs[str(protac_id)]
    write_sdf(mol_protac, path_protac_sdf)
    smi2sdf(smiles_linker, path_linker_sdf)


[14:03:02] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom 10 ignored
[14:03:02] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom 18 ignored
[14:03:02] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom 10 ignored
[14:03:02] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom 18 ignored
[14:03:02] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom 10 ignored
[14:03:02] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom 18 ignored
[14:03:03] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom 10 ignored
[14:03:03] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom 18 ignored
[14:03:03] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom 10 ignored
[14:03:03] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom 18 ignored
[14:03:03] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom

6107


100%|██████████| 5925/5925 [02:13<00:00, 44.49it/s] 


In [15]:
for i in range(1, 6108):
    path_protac_sdf = os.path.join('SDF', str(i) + '_protac.sdf')
    sup = Chem.SDMolSupplier(path_protac_sdf)
    mol = sup[0]
    smi = Chem.MolToSmiles(mol)

OSError: File error: Bad input file SDF\11_protac.sdf